In [ ]:
from datetime import datetime
semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
print(semana_atual)

### Gerando uma lista virtual de todas as relações EUtraFreqRelation que devem existir na rede (co-sites)

##### 1 - Pegar os dados de setores no MO EutranCellFDD e transformar numa lista de freq por site
##### 2 - Criar uma lista de setores para fazer o join com a lista acima e gerar a lista virtual

In [ ]:
import pandas as pd

arquivo_EutranCellFDD= 'C:/Camilo/' + 'EUtranCell_TDD_FDD'+ '_'+str(semana_atual)+'.txt'

EutranCellFDD= pd.read_csv(arquivo_EutranCellFDD, sep='\t' ,usecols=['UF', 'MUNICIPIO','eNB', 'CELL', 'administrativeState','operationalState', 'earfcndl'], encoding='ISO-8859-1')
EutranCellFDD= EutranCellFDD[(EutranCellFDD['administrativeState']=='UNLOCKED') & (EutranCellFDD['operationalState']=='ENABLED')] # considerando apenas setores desbloqueados

EutranCellFDD1= EutranCellFDD.groupby(['UF','MUNICIPIO','eNB', 'earfcndl'])['CELL'].count()
EutranCellFDD1= EutranCellFDD1.reset_index()
EutranCellFDD1= EutranCellFDD1.drop('CELL', axis=1)

EutranCellFDD1.head()

In [ ]:
setores= EutranCellFDD.loc[:, ['UF', 'eNB', 'CELL']]
setores.head()

In [ ]:
lista_virtual= pd.merge(EutranCellFDD1, setores, left_on=['UF', 'eNB'], right_on=['UF', 'eNB'],
                        how='inner')
lista_virtual= lista_virtual[['UF', 'MUNICIPIO', 'eNB','CELL', 'earfcndl']]

lista_virtual.shape

### Pegando os dados das relações 4G--4G do MO EUtranFreqRelation

A frequência efetiva **não deve ser inferida pelo nome** de `EUtranFreqRelationId`/`earfcndl`. Esses campos podem conter rótulos incompletos ou personalizados, como `LTE` e `LTE_700J`.

Fluxo adotado: extrair `EUtranFrequencyId` de `eutranFrequencyRef`, cruzar com `EUtranFrequency.txt` por `eNB + EUtranFrequencyId` e usar `arfcnValueEUtranDl` como EARFCN efetivo. Exemplos confirmados: `LTE` referencia `LTE_9460` (EARFCN 9460) e `LTE_700J` referencia um objeto com EARFCN 9310.

In [ ]:
import pandas as pd

chunk_size = 10000
chunks = []

colunas_relacao = ['UF', 'eNB', 'CELL', 'EUtranFreqRelationId', 'eutranFrequencyRef']

for chunk in pd.read_csv('C:/Dumps/LTE_total/EUtranFreqRelation.txt', delimiter='\t',
                         usecols=colunas_relacao, dtype='string',
                         chunksize=chunk_size, low_memory=False):
    # Extrai o objeto EUtranFrequency referenciado no Distinguished Name (DN).
    chunk['EUtranFrequencyId_ref'] = chunk['eutranFrequencyRef'].str.extract(
        r'vsDataEUtranFrequency=([^,]+)', expand=False
    )
    chunks.append(chunk)

EUtranFreqRelation = pd.concat(chunks, ignore_index=True)

EUtranFreqRelation.head()

In [ ]:
# Resolve o EARFCN pelo objeto EUtranFrequency realmente referenciado.
EUtranFrequency = pd.read_csv(
    'C:/Dumps/LTE_total/EUtranFrequency.txt', sep='\t',
    usecols=['eNB', 'EUtranFrequencyId', 'arfcnValueEUtranDl'],
    dtype='string', low_memory=False
)
EUtranFrequency = EUtranFrequency.drop_duplicates(subset=['eNB', 'EUtranFrequencyId'])

EUtranFreqRelation1 = pd.merge(
    EUtranFreqRelation, EUtranFrequency,
    left_on=['eNB', 'EUtranFrequencyId_ref'],
    right_on=['eNB', 'EUtranFrequencyId'],
    how='left', validate='m:1'
)
EUtranFreqRelation1['earfcndl_EUtranFreqRelation'] = pd.to_numeric(
    EUtranFreqRelation1['arfcnValueEUtranDl'], errors='coerce'
).astype('Int64')

# Impede falso missing se uma referência não puder ser resolvida.
referencias_sem_frequencia = EUtranFreqRelation1[
    EUtranFreqRelation1['earfcndl_EUtranFreqRelation'].isna()
]
if not referencias_sem_frequencia.empty:
    raise ValueError(
        f'{len(referencias_sem_frequencia)} relações sem frequência resolvida no MO EUtranFrequency'
    )

EUtranFreqRelation2 = EUtranFreqRelation1[[
    'UF', 'eNB', 'CELL', 'EUtranFreqRelationId',
    'EUtranFrequencyId_ref', 'earfcndl_EUtranFreqRelation'
]]

print(EUtranFreqRelation2['earfcndl_EUtranFreqRelation'].unique())

In [ ]:
missing= pd.merge(lista_virtual, EUtranFreqRelation2, left_on= ['UF', 'eNB', 'CELL', 'earfcndl'],
                 right_on=['UF', 'eNB', 'CELL', 'earfcndl_EUtranFreqRelation'], how='left')

missing1= missing[missing['earfcndl_EUtranFreqRelation'].isna()]
missing1= missing1.copy()
missing1['Regional']= missing1['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE', 'MG', 'SP'],
                                             ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE', 'MG', 'SP'])

missing1.shape

In [ ]:
missing1= missing1[['Regional', 'UF', 'MUNICIPIO', 'eNB','CELL','earfcndl','earfcndl_EUtranFreqRelation']]

missing2= missing1.sort_values(['Regional', 'UF', 'MUNICIPIO', 'eNB','CELL','earfcndl'], ascending=[True, True, True, True, True, True])

missing2= missing2[~missing2['earfcndl'].isin([2426, 1800, 549, 525])]

missing2

In [ ]:
missing2.to_excel('C:/Vivo Reports/LTE/LTE Mobilidade/Missing EUtranFreqRelation_cosite_4G_4G.xlsx', index=None)
print('OK')